# OAS dataset preparation for ESM2-35M evotuning

This notebook documents, end to end, the download → filter → dedup pipeline
that produces the OAS corpus used for evotuning, and *why* each step is done
the way it is. It implements item 2 of the `report/evotuning.md` "Next
experiments" checklist: *"Download, filter, and split OAS, then retrain the
35M model."* Every design decision below is tied back to a specific part of
`report/evotuning.md` — read that document first if you haven't; this
notebook assumes it as ground truth and doesn't re-argue its decisions,
only executes them.

Heavy steps (download, filtering, clustering) all run as separate SLURM jobs
(`bash_scripts/utils/{download_oas,filter_oas,linclust}.sbatch`) — the Euler
login node cannot handle this CPU/IO load directly. This notebook's code
cells are all cheap: they submit jobs, then read back job logs, directory
listings, and small samples to report and sanity-check what happened. No
cell in this notebook itself does heavy computation.

**Corpus location during this work:** everything lives under `$SCRATCH_DIR`
(`/cluster/scratch/mdenegri/protein-design/`) until the corpus is validated.
Promotion to `$PROJECT_DIR/data/oas/` (replacing the old corpus, which is
kept as `data/oas-old/`) is a separate, later step — not covered by the
cells below.

In [1]:
import os, subprocess, gzip
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env.local" if (Path.cwd().parent / ".env.local").exists() else ".env.local")

SCRATCH_DIR = Path(os.environ["SCRATCH_DIR"])
PROJECT_DIR = Path(os.environ["PROJECT_DIR"])
REPO = Path.cwd() if (Path.cwd() / "report").exists() else Path.cwd().parent

print(f"SCRATCH_DIR = {SCRATCH_DIR}")
print(f"PROJECT_DIR = {PROJECT_DIR}")
print(f"REPO        = {REPO}")

SCRATCH_DIR = /cluster/scratch/mdenegri/protein-design
PROJECT_DIR = /cluster/project/infk/krause/mdenegri/protein-design
REPO        = /cluster/home/mdenegri/protein-design


## Step 1 — Get the OAS bulk-download URL list

Per `report/evotuning.md` ("OAS filtering, download, deduplication, and
splits"): **"Download from OAS with only filters: human race and chain
heavy."** No isotype restriction (unlike the old corpus, which was
IGHG-only — a leftover of a narrower earlier query).

OAS's "Downloads" page is a red herring — it's static text pointing back at
the search UI. The actual bulk-download mechanism is the search page itself,
`https://opig.stats.ox.ac.uk/webapps/oas/oas_unpaired/`: a plain HTML POST
form. POSTing it with `Species=human`, `Chain=Heavy`, and every other field
wildcarded (`*`) returns an HTML page whose results table embeds one row per
matching data-unit *file* (not per data-unit) — each unit can appear as a
`_Bulk.csv.gz` (all isotypes combined) and/or as separate per-isotype files
(`_IGHA`, `_IGHD`, `_IGHE`, `_IGHG`, `_IGHM`). The per-isotype files are
subsets of their own `_Bulk` file, so downloading everything would mean
redundantly re-fetching the same sequences up to ~3.5x. We only want the
`_Bulk` file per unit.

In [2]:
import requests, re

resp = requests.post(
    "https://opig.stats.ox.ac.uk/webapps/oas/oas_unpaired/",
    files={
        "Species": (None, "human"), "BSource": (None, "*"), "BType": (None, "*"),
        "Longitudinal": (None, "*"), "Age": (None, "*"), "Disease": (None, "*"),
        "Subject": (None, "*"), "Vaccine": (None, "*"), "Chain": (None, "Heavy"),
        "Isotype": (None, "*"), "Primer": (None, "*"),
    },
    timeout=120,
)
resp.raise_for_status()
html = resp.text
print(f"HTTP {resp.status_code}, {len(html):,} bytes")

HTTP 200, 9,692,571 bytes


In [3]:
# Every row in the results table embeds the download URL plus a #Unique Sequences
# count -- parse both so we can quantify (not just count) the Bulk vs per-isotype split.
start = html.find('id="results_table"')
body = html[start:]
rows = re.findall(
    r'<tr>\s*<td><a href="\.\./dataunit\?unit=([^"]+)"[^>]*>Details</a></td>'
    r'\s*<td>([^<]*)</td>\s*<td>([^<]*)</td>.*?<td>([^<]*)</td>\s*<td>([^<]*)</td>',
    body, re.S,
)
print(f"Parsed {len(rows):,} result-table rows")

BASE = "https://opig.stats.ox.ac.uk/webapps/ngsdb/unpaired/"
bulk_urls, bulk_seqs, nonbulk_seqs = [], 0, 0
units_with_bulk, units_any = set(), set()
nonbulk_seqs_no_bulk_unit = 0

for unit, dsname, nseq, isotype, chain in rows:
    n = int(nseq)
    is_bulk = unit.endswith("_Heavy_Bulk.csv.gz")
    key = re.sub(r"_Heavy_(Bulk|IGH[ADEGM])\.csv\.gz$", "", unit)
    units_any.add(key)
    if is_bulk:
        bulk_urls.append(BASE + unit)
        bulk_seqs += n
        units_with_bulk.add(key)
    else:
        nonbulk_seqs += n

units_without_bulk = units_any - units_with_bulk
# sequences contributed only by units that have no Bulk file at all
for unit, dsname, nseq, isotype, chain in rows:
    key = re.sub(r"_Heavy_(Bulk|IGH[ADEGM])\.csv\.gz$", "", unit)
    if key in units_without_bulk and not unit.endswith("_Heavy_Bulk.csv.gz"):
        nonbulk_seqs_no_bulk_unit += int(nseq)

print(f"Distinct data-units (any file type): {len(units_any):,}")
print(f"Distinct data-units with a Bulk file: {len(units_with_bulk):,}")
print(f"Data-units WITHOUT a Bulk file:       {len(units_without_bulk):,}")
print(f"Bulk-file URLs:                       {len(bulk_urls):,}")
print()
print(f"Sequences in Bulk files:                         {bulk_seqs:,}")
print(f"Sequences in the {len(units_without_bulk)} Bulk-less units (isotype files only): {nonbulk_seqs_no_bulk_unit:,}")
print(f"  -> fraction of Bulk total: {nonbulk_seqs_no_bulk_unit / bulk_seqs:.5%}")

Parsed 13,265 result-table rows
Distinct data-units (any file type): 3,874
Distinct data-units with a Bulk file: 3,792
Data-units WITHOUT a Bulk file:       82
Bulk-file URLs:                       3,792

Sequences in Bulk files:                         502,394,549
Sequences in the 82 Bulk-less units (isotype files only): 72,576
  -> fraction of Bulk total: 0.01445%


**Decision: skip the Bulk-less units.** They contribute a negligible fraction
of the corpus (confirmed above, well under 0.1%) relative to the added
pipeline complexity of a second download path with a different filename
convention. This was confirmed with the project owner before proceeding
(2026-07-16).

Writing the final Bulk-only URL list to the repo root (gitignored, like the
old `oas_ighg_urls.txt` it replaces) — `bash_scripts/utils/download_oas.sbatch`
reads this file by name.

In [4]:
url_list_path = REPO / "oas_human_heavy_urls.txt"
print(f"URL list already written at: {url_list_path}")
print(f"Line count: {sum(1 for _ in open(url_list_path)):,}")
print("First 3 lines:")
for line in list(open(url_list_path))[:3]:
    print(" ", line.strip())

URL list already written at: /cluster/home/mdenegri/protein-design/oas_human_heavy_urls.txt


Line count: 3,792
First 3 lines:
  https://opig.stats.ox.ac.uk/webapps/ngsdb/unpaired/Bashford_2013/csv/ERR220430_Heavy_Bulk.csv.gz
  https://opig.stats.ox.ac.uk/webapps/ngsdb/unpaired/Bashford_2013/csv/ERR220397_Heavy_Bulk.csv.gz
  https://opig.stats.ox.ac.uk/webapps/ngsdb/unpaired/Bashford_2013/csv/ERR220429_Heavy_Bulk.csv.gz


## Step 2 — Download

```bash
sbatch bash_scripts/utils/download_oas.sbatch
```

Runs `scripts/data_prep/download_oas.sh oas_human_heavy_urls.txt`, a plain
sequential `wget` loop into `$SCRATCH_DIR/oas_raw/`, skipping files that
already exist (safe to re-run/resume). The old sbatch time budget (4h) was
sized for the narrower IGHG-only query; bumped to 24h here since the
Bulk-only query pulls substantially more data per unit.

In [5]:
result = subprocess.run(
    ["sacct", "-j", "7312727", "--format=JobID,JobName,State,ExitCode,Elapsed,MaxRSS", "--noheader"],
    capture_output=True, text=True,
)
print(result.stdout)

log_path = REPO / "bash_scripts/logs/oas_download_7312727.log"
tail = subprocess.run(["tail", "-c", "500", str(log_path)], capture_output=True, text=True).stdout
print("--- end of download log ---")
print(tail)

n_files = len(list((SCRATCH_DIR / "oas_raw").glob("*.csv.gz")))
du = subprocess.run(["du", "-sh", str(SCRATCH_DIR / "oas_raw")], capture_output=True, text=True).stdout
print(f"\nFiles in oas_raw/: {n_files:,} (expected 3792)")
print(f"Total size: {du.strip()}")

7312727      oas_downl+  COMPLETED      0:0   01:16:36            
7312727.bat+      batch  COMPLETED      0:0   01:16:36   8377520K 
7312727.ext+     extern  COMPLETED      0:0   01:16:36            

--- end of download log ---
..... 90% 84.5M 0s
  2400K .......... .......... .......... .......... .......... 92%  100M 0s
  2450K .......... .......... .......... .......... .......... 94%  101M 0s
  2500K .......... .......... .......... .......... .......... 96%  102M 0s
  2550K .......... .......... .......... .......... .......... 98% 87.5M 0s
  2600K .......... .......... .......... .......... .......   100%  112M=0.2s[OK]   SRR924017_1_Heavy_Bulk.csv.gz

Done. Total: 3792 | Downloaded: 3792 | Skipped: 0 | Failed: 0




Files in oas_raw/: 3,792 (expected 3792)
Total size: 231G	/cluster/scratch/mdenegri/protein-design/oas_raw
